In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, linregress
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
DIAGNOSTICS = INPUT/'diagnostics'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
MODELS = CONFIG['models']
PRIMARY = CONFIG['primary_plot_models']
LABELS = {'arw4':'ARW4', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'AR(1)-H', 'ar6_6':'AR(6,6)', 'se_hurdle_s':'SE-H'}
SECONDARY = OUTPUT/'secondary'
SECONDARY.mkdir(exist_ok=True)
VIEWS = [(OUTPUT,PRIMARY), (SECONDARY,MODELS)]
palette = sns.color_palette('colorblind',6)
COLORS = {source:palette[i] for source,i in zip(LABELS,[3,0,1,4,5,2])}
MARKERS = dict(zip(LABELS,['P','o','s','^','D','X']))
DASHES = dict(zip(LABELS,[(3,1,1,1),'',(4,2),(2,1),(6,2,1,2),(8,2)]))

In [ ]:
panel = pd.read_csv(DIAGNOSTICS/'simulation/fit/data/panel_complete.csv',index_col='dblp_id')
observed = panel.to_numpy(dtype=float).T
models = {source:np.load(DIAGNOSTICS/'simulation'/source/'trajectories.npy',mmap_mode='r') for source in MODELS}
generated_from = (pd.read_csv(DIAGNOSTICS/'simulation_metadata.csv')
                  .set_index('model')['generated_from_age'].astype(int))

In [ ]:
# exact W1 on the raw annual output samples, including zeros
rows = []
for source,array in models.items():
    for age in range(Y+1):
        rows.append({'source':source,'age':age,'n_empirical':observed.shape[1],'n_simulated':array.shape[1],
                     'generated_from_age':generated_from[source],'is_generated':age>=generated_from[source],
                     'wasserstein1':wasserstein_distance(observed[age],array[age])})
distances = pd.DataFrame(rows)
distances.to_csv(OUTPUT/'annual_wasserstein.csv',index=False)

In [ ]:
# equally weighted mean and raw-scale OLS trend over all ages
rows = []
for source,frame in distances.groupby('source',sort=False):
    fit = linregress(frame.age,frame.wasserstein1)
    rows.append({'source':source,'start_age':int(frame.age.min()),'end_age':int(frame.age.max()),
                 'n_years':len(frame),'n_initialized_years':int((~frame.is_generated).sum()),
                 'mean_wasserstein1':frame.wasserstein1.mean(),'slope_wasserstein1':fit.slope})
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT/'annual_wasserstein_summary.csv',index=False)
display(summary)

In [ ]:
sns.set_theme(style='ticks',context='paper',rc={'pdf.fonttype':42})
for dest,selected in VIEWS:
    fig,ax = plt.subplots(figsize=(7.2,4.1))
    if 'ar6_6' in selected:
        ax.axvspan(0, generated_from['ar6_6']-.5, color='.9', zorder=0, label='AR(6,6) initialized')
    ax.axhline(0,color='.3',lw=.8,ls=':',label='DBLP reference')
    sns.lineplot(data=distances.loc[distances.source.isin(selected)],x='age',y='wasserstein1',
                 hue='source',style='source',hue_order=selected,style_order=selected,
                 palette=COLORS,markers=MARKERS,dashes=DASHES,markersize=4,
                 estimator=None,errorbar=None,ax=ax)
    handles,names = ax.get_legend_handles_labels()
    ax.legend(handles,[LABELS.get(name,name) for name in names],frameon=False,ncol=2,title=None)
    ax.set(xlabel='Career year',ylabel='Wasserstein-1 (papers)',xlim=(0,Y),
           title='Annual productivity distance to DBLP')
    ax.set_xticks(range(0,Y+1,2))
    ax.set_yscale('symlog',linthresh=1)
    ax.set_ylim(bottom=-.06)
    sns.despine(ax=ax)
    for ext in ['pdf','png']:
        fig.savefig(dest/f'annual_wasserstein.{ext}',bbox_inches='tight',dpi=180,facecolor='white')
    plt.close(fig)